In [3]:
import torch
import torch.nn as nn
x = nn.Conv2d(
    3,
    768, kernel_size = 16,
    stride = 16,
    padding = 0
)
img = torch.randn(3,224,224)
img_dim = img.shape[-1]
img = img.unsqueeze(0)
y =x(img).flatten(2).permute(0,2,1)
img.shape,y.shape

(torch.Size([1, 3, 224, 224]), torch.Size([1, 196, 768]))

In [4]:
(torch.Size([1, 3, 224, 224]), torch.Size([1, 768, 196]))

(torch.Size([1, 3, 224, 224]), torch.Size([1, 768, 196]))

In [5]:
x(img).shape

torch.Size([1, 768, 14, 14])

In [6]:
import torch
import torch.nn as nn
m = nn.Parameter(torch.randn(1,4))

In [7]:
x = nn.Linear(4,1).weight

In [8]:
x.shape

torch.Size([1, 4])

In [9]:
x

Parameter containing:
tensor([[ 0.4694, -0.2291,  0.2673, -0.4900]], requires_grad=True)

In [10]:
m

Parameter containing:
tensor([[-0.4157,  1.9440, -0.0043,  0.8885]], requires_grad=True)

# Ok Let's understand VIT

## The Inputs (Patch , positional and cls token)

In [11]:
class PatchEmbedding(nn.Module):
    def __init__(self, config):
        self.image_size = config['image_size']
        self.patch_size = config['patch_size']
        self.num_channels = config['num_channels']
        self.hidden_size = config['hidden_size']
        self.num_patches = (self.image_size // self.patch_size)**2
        self.projection = nn.Conv2d(self.num_channels, self.hidden_size, self.patch_size, self.patch_size)

    
    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(x).transpose(1,2)
        return x

In [12]:
class Embeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.patch_embeddings = PatchEmbedding(config)
        self.cls_token = nn.Parameter(torch.randn(1, 1, config['hidden_dim']))
        self.positional_embedding = nn.Parameter(torch.randn(1, self.patch_embeddings.num_patches +1, config['hidden_dim']))
        self.dropout = nn.Dropout(config['hidden_dropout_rate'])

    
    def forward(self, x):
        x = self.patch_embeddings(x)
        batch_size, _, _ = x.size()
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat((cls_tokens, x), dim = 1)
        x = x+ self.positional_embedding
        x = self.dropout(x)
        return x

In [13]:
mm =  nn.Parameter(torch.randn(1, 1,8))
z = mm.expand(2, -1, -1)

## Rough

In [14]:
mm

Parameter containing:
tensor([[[-0.5183,  1.7871, -1.8330,  1.4334, -0.1141, -0.3792, -0.1452,
          -0.3388]]], requires_grad=True)

In [15]:
z.shape

torch.Size([2, 1, 8])

In [16]:
z

tensor([[[-0.5183,  1.7871, -1.8330,  1.4334, -0.1141, -0.3792, -0.1452,
          -0.3388]],

        [[-0.5183,  1.7871, -1.8330,  1.4334, -0.1141, -0.3792, -0.1452,
          -0.3388]]], grad_fn=<ExpandBackward0>)

In [17]:
x = torch.randn(2, 4, 8)

In [18]:
d = torch.cat((z, x), dim = 1)

In [19]:
d.shape

torch.Size([2, 5, 8])

## MHA

In [20]:
import math
class AttentionHead(nn.Module):
    def __init__(self, hidden_size, attention_head_size, dropout, bias = True):
        super().__init__()
        self.hidden_size = hidden_size #768
        self.attention_head_size = attention_head_size #192
        self.query = nn.Linear(hidden_size, attention_head_size, bias = bias) #(768, 192)
        self.key = nn.Linear(hidden_size, attention_head_size, bias=bias) #(768, 192)
        self.value = nn.Linear(hidden_size, attention_head_size, bias=bias) #(768, 192)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        #x = (batch_size, number of patches, hidden_dim) -> (4, 196, 768)
        query = self.query(x) # (4, 196, 192)
        key = self.key(x) # (4, 196, 192)
        value = self.value(x) # (4, 196, 192)
        attention_weights = torch.matmul(query, key.transpose(-2,-1)) #(4, 196, 196)
        attention_weights = attention_weights / math.sqrt(self.attention_head_size) #(4, 196, 196)
        attention_scores = nn.functional.softmax(attention_weights, dim = -1) #(4, 196, 196)
        attention_scores = self.dropout(attention_scores) #(4, 196, 196)
        attention_output = torch.matmul(attention_scores, value) #(4, 196, 196) * (4, 196, 192) = ( 4, 196, 192)
        return (attention_output, attention_scores) #( [4,196, 192], [4, 196, 196])


In [ ]:
class MHA(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config['hidden_size'] #768
        self.num_attention_heads = config['num_attention_head'] # 4
        self.attention_head_size = self.hidden_size // self.num_attention_heads # 192
        self.all_head_size = self. num_attention_heads* self.attention_head_size #768
        self.qkv_bias = config['qkv_bias'] 
        self.heads = nn.ModuleList([])
        for _ in range(self.num_attention_heads):
            head =AttentionHead(self.hidden_size,
                                self.attention_head_size,
                                 config['attention_prob_dropout_rate'],
                                  self.qkv_bias)
            self.heads.append(head)

        self.output_projection = nn.Linear(self.all_head_size, self.hidden_size) #(768, 768)
        self.output_dropout = nn.Dropout(config['hidden_dropout_rate']) 


    def forward(self, x, output_attentions = False):
        attention_outputs = [head(x) for head in self.heads] #[([4,196, 192], [4, 196, 196]), ([4,196, 192], [4, 196, 196]),([4,196, 192], [4, 196, 196]), ([4,196, 192], [4, 196, 196])]
        attention_output = torch.cat([attention_output for attention_output, _ in attention_outputs], dim = -1) # [[4,196, 192],[4,196, 192], [4,196, 192] , [4,196, 192]] -> [4, 196, 768]
        attention_output = self.output_projection(attention_output)   #(4, 196, 768) * (768, 768) -> (4, 196, 768)
        attention_output = self.output_dropout(attention_output) (4, 196, 768)

        if not output_attentions:
            return (attention_output, None)
        else:
            attention_probs = torch.stack([attention_probs for _, attention_probs in attention_outputs], dim = 1) # [( 4, 196, 196)*4] -> (4, 4, 196, 196)
            return (attention_output, attention_probs) # ([4, 196, 768], [4, 4, 196, 196] )


## MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense_1 = nn.Linear(config['hidden_size'], config['intermediate_size']) #(768, 2000)
        self.activation = nn.functional.gelu()
        self.dense_2 = nn.Linear(config['intermediate_size'], config['hidden_size']) #(2000, 768)
        self.dropout = nn.Dropout(config['hidden_dropout_rate'])

    def forward(self, x):
        x = self.dense_1(x) #(4, 196, 2000)
        x = self.activation(x) #(4, 196, 2000)
        x = self.dense_2(x) #(4, 196, 768)
        x = self.dropout(x) #(4, 196, 768)
        return x #(4, 196, 768)

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MHA(config)
        self.layernorm_1 = nn.LayerNorm(config['hidden_size'])

        self.mlp = MLP(config)
        self.layernorm_2 = nn.LayerNorm(config['hidden_size'])

    def forward(self, x, output_attentions = False):
        attention_output, attention_probs = self.attention(self.layernorm_1(x), output_attentions = output_attentions) # ([4, 196, 768], [4, 4, 196, 196] )
        x = x + attention_output #(4, 196, 768) + (4, 196, 768) -> (4, 196, 768)
        mlp_output = self.mlp(self.layernorm_2(x)) #(4, 196, 768)
        x = x + mlp_output #(4, 196, 768)
        if not output_attentions:
            return (x, None) #((4, 196, 768), None)
        else:
            return (x, attention_probs) #((4, 196, 768), (4, 4, 196, 196))

In [ ]:
class Encoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn. ModuleList([])
        for _ in range(config['num_hidden_layers']):
            block = Block(config)
            self.blocks.append(block)


    def forward(self, x, output_attentions = False):
        all_attentions = []
        for block in self.blocks:
            x, attention_probs = block(x, output_attentions = output_attentions)
            if output_attentions:
                all_attentions.append(attention_probs)

        if not output_attentions:
            return (x, None) #((4, 196, 768), None)
        
        else:
            return (x, all_attentions) #((4, 196, 768), [[4, 4, 196, 196]]*4)

In [38]:
class ViTforClassification(nn. Module):
    def __init__(self, config):
        super().__init__()
        self.config = config 
        self.image_size = config['image_size']
        self.hidden_size = config['hidden_size']
        self.num_classes =  config['num_classes']
        self.embedding = Embeddings(config)
        self.encoder = Encoder(config)
        self.classifier = nn.Linear(self.hidden_size, self.num_classes)
        self.apply(self.__init__weights)

    def forward(self, x, output_attentions = False):
        embedding_output = self.embedding(x)
        encoder_output, all_attentions = self.encoder(embedding_output, output_attentions = output_attentions)
        logits = self.classifier(encoder_output[:, 0])

        if not output_attentions:
            return (logits, None)
        else:
            return (logits, all_attentions)


In [37]:
x = torch.randn(4, 196, 768)
x[:,0].shape
# m = torch.cat((x, x, x), 0)
# n = torch.cat((x, x, x), 1)

torch.Size([4, 768])

In [31]:
x = torch.randn(2, 3)
x
print(torch.stack((x, x))) # same as torch.stack((x, x), dim=0)
print(torch.cat((x, x)))
print(torch.cat((x, x)).shape)
print(torch.stack((x, x)).size())
print(torch.stack((x, x), dim=1))
print(torch.stack((x, x), dim=1).shape)

print(torch.stack((x, x), dim=-1))
print(torch.stack((x, x), dim=2))


tensor([[[ 1.0767,  0.2579,  0.1431],
         [-1.4330, -0.4373, -0.2006]],

        [[ 1.0767,  0.2579,  0.1431],
         [-1.4330, -0.4373, -0.2006]]])
tensor([[ 1.0767,  0.2579,  0.1431],
        [-1.4330, -0.4373, -0.2006],
        [ 1.0767,  0.2579,  0.1431],
        [-1.4330, -0.4373, -0.2006]])
torch.Size([4, 3])
torch.Size([2, 2, 3])
tensor([[[ 1.0767,  0.2579,  0.1431],
         [ 1.0767,  0.2579,  0.1431]],

        [[-1.4330, -0.4373, -0.2006],
         [-1.4330, -0.4373, -0.2006]]])
torch.Size([2, 2, 3])
tensor([[[ 1.0767,  1.0767],
         [ 0.2579,  0.2579],
         [ 0.1431,  0.1431]],

        [[-1.4330, -1.4330],
         [-0.4373, -0.4373],
         [-0.2006, -0.2006]]])
tensor([[[ 1.0767,  1.0767],
         [ 0.2579,  0.2579],
         [ 0.1431,  0.1431]],

        [[-1.4330, -1.4330],
         [-0.4373, -0.4373],
         [-0.2006, -0.2006]]])


In [ ]:
m, n

(tensor([[-1.6558,  1.6391,  1.0757],
         [-1.6043,  1.0004, -0.7274],
         [-1.6558,  1.6391,  1.0757],
         [-1.6043,  1.0004, -0.7274],
         [-1.6558,  1.6391,  1.0757],
         [-1.6043,  1.0004, -0.7274]]),
 tensor([[-1.6558,  1.6391,  1.0757, -1.6558,  1.6391,  1.0757, -1.6558,  1.6391,
           1.0757],
         [-1.6043,  1.0004, -0.7274, -1.6043,  1.0004, -0.7274, -1.6043,  1.0004,
          -0.7274]]))